# GroupXX Midterm: Panorama Image Stitching (Proposal + V0)

CSC14120 Parallel Programming. Team: NDB (lead), LCK, DHDH.

> **TODO summary (Stage 1).** Every cell below starts with its TODO: owner, week/day, and what goes in it.
> Rules: the notebook builds from the submitted `src/` (never a copy pasted in a cell), runs with
> Runtime -> Run all on a fresh Colab GPU session, and every number in the text comes from an output cell.
>
> - TODO(NDB, Week 3 Day 20): replace `XX` with our group number in this title and in the file name.

## 0. Setup

The first cell gets the code and builds it, the second gets the dataset, the third defines the helpers used by the result cells.
The code comes from the folder next to this notebook if `src/` is there (submission unzipped), otherwise from the team's shared Drive folder.

In [ ]:
# Setup (NDB). Works with Runtime -> Run all on a fresh Colab GPU session.
import os, shutil
from pathlib import Path

DRIVE_PROJECT = '/content/drive/MyDrive/CSC14120-Final-Project'  # shared Drive copy of the repo
WORK = Path('/content/project')

if Path('src/main.cu').exists() and Path('Makefile').exists():
    WORK = Path.cwd()                      # submission unzipped next to the notebook
else:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK.mkdir(parents=True, exist_ok=True)
    for name in ['src', 'tools']:
        shutil.copytree(f'{DRIVE_PROJECT}/{name}', WORK / name, dirs_exist_ok=True)
    shutil.copy(f'{DRIVE_PROJECT}/Makefile', WORK / 'Makefile')
os.chdir(WORK)
print('Working folder:', WORK)

!nvcc --version | tail -n 2
!make
!nvidia-smi -L

In [ ]:
# Dataset (NDB). Link from LCK (Week 1 Day 5), also given in section 3.6.
#   1) ./dataset already there -> use it
#   2) the zip on the mounted shared Drive -> unzip it
#   3) otherwise download the zip from the public link with gdown (for a fresh session)
import subprocess, zipfile

DATASET_ZIP = f'{DRIVE_PROJECT}/dataset.zip'
DATASET_DRIVE_ID = ''  # TODO(LCK, Week 1 Day 5): file id of dataset.zip on the shared Drive (public link)
SIZES = ['small', 'medium', 'large']

if not Path('dataset').exists():
    if not Path(DATASET_ZIP).exists() and DATASET_DRIVE_ID:
        subprocess.run(['gdown', DATASET_DRIVE_ID, '-O', 'dataset.zip'], check=True)
        DATASET_ZIP = 'dataset.zip'
    if Path(DATASET_ZIP).exists():
        with zipfile.ZipFile(DATASET_ZIP) as z:
            z.extractall('.')
    else:
        print('Dataset not found: set DATASET_ZIP or DATASET_DRIVE_ID above.')

# Pairs = every dataset/small/<pair>_A.png that has a matching _B.png.
PAIRS = sorted(p.name[:-len('_A.png')] for p in Path('dataset/small').glob('*_A.png')
               if (p.parent / p.name.replace('_A.png', '_B.png')).exists())
print('Pairs:', PAIRS)
for size in SIZES:
    print(f'{size:7s}', len(list(Path('dataset', size).glob('*.png'))), 'images')

In [ ]:
# Helpers for the result cells (NDB): run ./stitch once and read what it prints.
import re

def run_stitch(a, b, out, runs=1, debug=None):
    """Run V0 on (a, b), write the panorama to out, return the program output (text)."""
    Path(out).parent.mkdir(parents=True, exist_ok=True)
    cmd = ['./stitch', str(a), str(b), str(out), '--version', '0', '--runs', str(runs)]
    if debug:
        cmd += ['--debug', str(debug)]
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout, r.stderr)
        r.check_returncode()
    return r.stdout

def parse_counts(log):
    """Numbers from the 'V0 counts' line, e.g. {'keypoints A': 2000, ..., 'inliers': 874}."""
    line = next(l for l in log.splitlines() if l.startswith('V0 counts'))
    return {'keypoints A': int(re.search(r'keypoints A=(\d+)', line).group(1)),
            'keypoints B': int(re.search(r' B=(\d+)', line).group(1)),
            'matches': int(re.search(r'matches=(\d+)', line).group(1)),
            'inliers': int(re.search(r'inliers=(\d+)', line).group(1))}

## 3.1 Team information and work plan

| Member | Full name | Student ID |
|---|---|---|
| NDB (lead) | TODO(NDB) | TODO(NDB) |
| LCK | TODO(LCK) | TODO(LCK) |
| DHDH | TODO(DHDH) | TODO(DHDH) |

**Who does what, until the final submission.** Each member owns two pipeline stages in every version: the same person writes the sequential code and later the CUDA kernels of those stages, and one optimized version.

| Member | Stages (V0, then V1) | Optimized version (Stage 2) | Shared work |
|---|---|---|---|
| NDB | 1. Preprocessing, 2. Keypoint detection | V3: shared-memory tiles (with halo) for blur and Harris, blur weights in constant memory | Git repo, build and timing harness, integration, notebook setup, README (final), submission |
| LCK | 3. Description (BRIEF), 4. Matching | V2: B descriptors in shared memory + `__popc` in matching | Dataset preparation, benchmark script, timing tables and charts, Amdahl estimate, slides |
| DHDH | 5. Homography (RANSAC), 6. Warping + blending | V4: keep data on the GPU between stages + pinned memory (optional: 2 streams for A and B) | Synthetic pair test, panorama gallery, verify tool (V1+ vs V0), README (midterm), video editing |

Change to the plan in Stage 1: NDB wrote the V0 code of stages 5–6 (`v0_homography.cpp`, `v0_warp.cpp`, `ransac_samples.cpp`) and the synthetic-pair tools for DHDH; DHDH reviewed them and owns them from then on (CUDA versions included).

**Timeline.**

| Period | Goal | Milestone |
|---|---|---|
| Stage 1, week 1 (Day 1–7) | Repo, shared data types, harness, photos of 3 sites + 1 challenging pair, first V0 code | Check-in, Day 7 |
| Stage 1, week 2 (Day 8–14) | Finish all 6 V0 stages, integrate, test on all pairs | Tag `v0-midterm`, Day 14 |
| Stage 1, week 3 (Day 15–20) | Timing, synthetic test, panoramas, GPU plan, this notebook | Midterm submission, Day 20 |
| Stage 2, week 1 (Day 1–7) | V1: every stage on the GPU with simple kernels; verify tool | Tag `v1`, Day 7 |
| Stage 2, week 2 (Day 8–14) | Profile V1, then V2, V3, V4 (order confirmed by the profile on Day 9) | Tags `v2`–`v4`, code freeze Day 14 |
| Stage 2, week 3 (Day 15–20) | Final notebook, README, slides, video, defence practice | Final submission, Day 20 |

## 3.2 Problem understanding

**Input:** two photos of the same scene, A (left) and B (right), taken from one place by turning the camera, with 30–50% overlap and the same zoom. **Output:** one wider image (panorama) that shows both.

**Why one homography is enough.** If the camera only turns around its optical center, a point of B and the point of A that see the same direction are linked by a 3x3 projective transform H, whatever the depth of the scene: $[x_A, y_A, 1]^T \sim H\,[x_B, y_B, 1]^T$. H has 8 unknowns (we fix $h_{33} = 1$), so 4 point pairs are enough to find it. If the photographer walked between the shots, near objects would move more than far ones (parallax) and no single H could fit both; that is why our photos are taken by rotating, not walking.

**The pipeline.** To find H we need point pairs that show the same thing in A and B. Stages 1–3 look at each image on its own (so they run on A and on B); stages 4–6 need both.

| Stage | Input | Output | Runs on |
|---|---|---|---|
| 1. Preprocessing | RGB image (w x h x 3 bytes) | blurred gray image (w x h floats) | A, B |
| 2. Keypoint detection | blurred gray image | up to K = 2000 corners (x, y, score), best first | A, B |
| 3. Description | blurred gray image + corners | a 256-bit code per corner (corners too close to the border are dropped) | A, B |
| 4. Matching | the codes of A and of B | pairs (corner of A, corner of B) that are clearly the best match | pair |
| 5. Homography | matched corners | H (B → A) and its number of inliers | pair |
| 6. Warping + blending | A, B, H | the panorama | pair |

In words: we make the image gray and remove noise (1), find corners, which are easy to find again in the other photo (2), describe the small patch around each corner with a short binary code (3), pair each corner of A with the corner of B that has the most similar code (4), find the H that most pairs agree with, so wrong pairs are ignored (5), and finally draw B in A's frame and fade one image into the other where they overlap (6).

**Where the work is.** Stages 1, 2 and 6 do the same work for every pixel, so their time grows with the image size. Stage 3 grows with the number of corners (at most K per image). Stage 4 compares every corner of A with every corner of B, up to K x K = 4 million code comparisons, whatever the image size. Stage 5 does 1000 iterations, each testing all matches. The measured split is in 3.5.3.

## 3.3 Chosen methods

TODO(DHDH, Week 3 Day 17): one table with these columns: stage, method, parameters, why (quality), why (GPU). Build it from `docs/method_notes.md`, which the stage owners write: LCK and DHDH in Week 1 Day 2, NDB in Week 2 Day 10. The parameters must match `src/common.h`.

## 3.4 Sequential version (V0)

TODO(owner not set in the plan; decide at the Week 1 check-in), Week 3:
- Design: one file per stage (`src/v0_*.cpp`), the shared interfaces (`common.h`, `v0.h`), the CLI, and how timing works (1 warm-up + N runs, `std::chrono` per stage, I/O not timed).
- State that the code is single-threaded: no OpenMP, no `std::thread`, no hand-written SIMD.
- Only short code excerpts, where they explain a design choice.
- Cite any adapted code here and in the source file.

## 3.5 V0 results and profiling

### 3.5.1 Correctness: homography error on the synthetic pair

TODO(DHDH, Week 3 Day 15): make the synthetic pair (`tools/make_synthetic_pair.py`), run `./stitch`, then compute the mean corner error against the true H (`tools/homography_error.py`). Do this for the 3 sizes and show one small table.

In [ ]:
# TODO(DHDH, Week 3 Day 15): synthetic pair test
#   for size in small, medium, large:
#     !python tools/make_synthetic_pair.py dataset/<size>/<pair>_A.png dataset/synthetic/<size>
#     !./stitch .../synth_A.png .../synth_B.png ... --version 0 --runs 1 > run_<size>.log
#     !python tools/homography_error.py .../H_true.txt run_<size>.log --width W --height H
print("TODO(DHDH, Week 3 Day 15): synthetic pair test")

### 3.5.2 Panoramas

TODO(DHDH, Week 2 Day 14 to make them, Week 3 Day 16 for the gallery): one panorama per site, plus the challenging pair, all at medium size.

In [ ]:
# TODO(DHDH, Week 3 Day 16): run ./stitch on each pair (medium) and show the panoramas
#   with a short caption each (site, inliers printed by the program).
print("TODO(DHDH, Week 3 Day 16): panorama gallery")

### 3.5.3 Timing per stage

TODO(LCK, Week 3 Day 15–16): run `tools/run_benchmark.py` (3 sizes x 4 pairs, 5 runs each) and show:
- a per-stage timing table for each size;
- a stacked bar chart, one bar per size, with a segment per stage.

In [ ]:
# TODO(LCK, Week 3 Day 15): !python tools/run_benchmark.py --stitch ./stitch --dataset dataset --out results/timing_v0.csv
print("TODO(LCK, Week 3 Day 15): run the benchmark")

In [ ]:
# TODO(LCK, Week 3 Day 16): load results/timing_v0.csv with pandas, print the per-stage table
#   (mean over the 4 pairs, per size), and draw the stacked bar chart with matplotlib.
print("TODO(LCK, Week 3 Day 16): timing table + stacked bar chart")

### 3.5.4 Hot stages

TODO(LCK, Week 3 Day 16): which stages take the most time, and how that changes from small to large. Quote the numbers from the output cells above.

### 3.5.5 Amdahl estimate

TODO(LCK, Week 3 Day 17): p = the fraction of V0 time spent in the stages we move to the GPU, measured at medium size. The end-to-end speedup limit is 1 / (1 - p). Compute both in the code cell, then explain them here.

In [ ]:
# TODO(LCK, Week 3 Day 17): compute p and 1 / (1 - p) per size from results/timing_v0.csv
print("TODO(LCK, Week 3 Day 17): Amdahl estimate")

## 3.6 Dataset

TODO(DHDH, Week 3 Day 16): the 3 sites (indoor, outdoor, team choice) and the challenging pair: where, when, who took them, and what makes the challenging pair hard. Explain the preparation (16:9 crop, 3 sizes). Include the shared Drive link to the raw photos + `dataset/` (LCK, Week 1 Day 5).

In [ ]:
# TODO(DHDH, Week 3 Day 16): show A and B of every pair side by side (small size thumbnails)
print("TODO(DHDH, Week 3 Day 16): dataset thumbnails")

## 3.7 GPU plan

For each stage: why it goes to the GPU (from the timing in 3.5.3), the V1 thread mapping, and later optimization ideas with their risks. V1 keeps the V0 methods, parameters and output order, so its results can be checked against V0.

### Stages 1–2: preprocessing, detection

**Why the GPU.** Every step of these two stages is the same small computation for every pixel, with no dependency between pixels, and the work grows with the image size (see how their share changes from small to large in 3.5.3–3.5.4). Only the last step (collecting the corners and keeping the best K) is not per pixel.

**V1 mapping** (2D blocks of 32x8 threads, one thread per pixel unless noted; data in global memory):

| Step | One thread computes | Notes |
|---|---|---|
| Grayscale | 1 gray value from 3 bytes | coalesced: neighbor threads read neighbor pixels |
| Blur, row pass | 1 pixel: 5 weighted neighbors in its row | clamp-to-edge exactly as V0 |
| Blur, column pass | 1 pixel: 5 weighted neighbors in its column | |
| Sobel | Ix and Iy of 1 pixel | |
| Harris | products Ix², Iy², IxIy, then row sums, then column sums + R | 3 kernels, same order of sums as V0 |
| NMS | the 3x3 test of 1 pixel; a winner reserves a slot with `atomicAdd` on a counter and writes (score, x, y) | atomics give a random order |
| Top K | `thrust::sort` by (score desc, y, x), keep the first K | same key as V0 → same list as V0 |

**Later ideas (V3) and their risks.**
1. *Shared-memory tile with a halo.* A block loads its tile plus a 2-pixel border once (blur), or a 3-pixel border (Sobel + 5x5 window sums), and all threads read neighbors from shared memory instead of reading each global pixel 5–25 times. Risk: the halo at the image border must clamp exactly like V0; a large halo means more shared memory per block, which can lower occupancy.
2. *Blur weights in constant memory.* All threads of a warp read the same weight at the same time, which constant memory broadcasts. Risk: small gain (only 5 weights).
3. *Fusing kernels* (gray + row blur; Sobel + products). Fewer writes and reads of whole images in global memory. Risk: more registers per thread, and the intermediate images cannot be compared with V0 any more.

**Correctness risk.** The GPU may use fused multiply-add and another order of additions, so R can differ from V0 in the last bits; two corners with almost equal scores near the K-th place can then swap. We will compare the keypoint lists with V0 (count, share of identical positions) and, if needed, check with `--fmad=false`.

### Stages 3–4: description, matching

TODO(LCK, Week 3 Day 18)

### Stages 5–6: homography, warping

TODO(DHDH, Week 3 Day 18)

## References and AI tool declaration

TODO(all): add your own sources and AI use; NDB checks on Week 3 Day 19.

**Papers, books, tutorials**
- C. Harris, M. Stephens. *A Combined Corner and Edge Detector.* Alvey Vision Conference, 1988. (stage 2)
- M. Calonder, V. Lepetit, C. Strecha, P. Fua. *BRIEF: Binary Robust Independent Elementary Features.* ECCV 2010. (stage 3)
- D. Lowe. *Distinctive Image Features from Scale-Invariant Keypoints.* IJCV 60(2), 2004 — ratio test. (stage 4)
- M. Fischler, R. Bolles. *Random Sample Consensus.* Communications of the ACM 24(6), 1981. (stage 5)
- R. Szeliski. *Computer Vision: Algorithms and Applications*, 2nd ed., 2022 — ch. 3 (filtering, resampling), ch. 7 (features), ch. 8 (alignment, stitching).
- ITU-R Recommendation BT.601 — luma weights for grayscale. (stage 1)
- TODO(LCK): https://www.youtube.com/watch?v=KgsHoJYJ4S8 (cited in `v0_describe.cpp`), https://paulinternet.nl/bicubic (cited in `script/resize`) — say what each was used for.

**Code**
- stb_image v2.30 and stb_image_write v1.16 by Sean Barrett, https://github.com/nothings/stb (public domain / MIT): image loading and saving (`src/stb_impl.cpp`).
- No other external code is copied. Code based on a paper or book cites it in the file header.

**AI tools**

| Member | Tool | Used for |
|---|---|---|
| NDB | Claude Code (Anthropic) | Drafting the harness, V0 stages 1–2, the V0 code of stages 5–6 and the synthetic-pair tools (written for DHDH), and notebook text. Every file was reviewed by NDB; stages 5–6 also by DHDH. |
| LCK | ChatGPT (OpenAI) | Bug fixing in `script/resize` (from its file header). TODO(LCK): complete. |
| DHDH | TODO(DHDH) | TODO(DHDH) |

## Submission checklist

- TODO(all, Week 3 Day 19, review day): fresh Colab GPU session, then Runtime -> Run all. Check that every number in the text matches an output cell and that the dataset link opens.
- TODO(NDB, Week 3 Day 20): save with all outputs, then `make submission GROUP=<n>` builds `Group<n>_Midterm.zip` (this notebook renamed + `src/` + `Makefile` + `tools/`, which the setup and result cells need). Submit (NDB only).